# PPO Implementation

In [1]:
import os
import sys
import time
import random
import numpy as np
import matplotlib.pyplot as plt

nb_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(nb_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)


SEED = 42
np.random.seed(SEED)
random.seed(SEED)
print('Using project root:', project_root)

Using project root: /Users/rithvik/Documents/hnrs/Decoder


In [2]:
from ldpc.bp_decoder import BpDecoder
from utils.LDPC_encode import QCLDPCEncoder
from utils.awgn_channel import AWGNChannel
from utils.find_ber import findBER
from ppo.ppo_env import PpoEnv
from ppo.ppo_agent import PpoAgent
from ppo.ppo_decoder import PpoDecoder

In [3]:
H = np.loadtxt(os.path.join(project_root, 'pc_matrices', 'WRAN_irreg_384_256.csv'), delimiter=',', dtype=int)

encoder = QCLDPCEncoder(H=H)
m, n = H.shape
k = n - m

clusters = np.arange(m).reshape(8, -1)

print(f'H shape: {H.shape}, K={encoder.K}, rate={encoder.K / encoder.N:.3f}')
print('Clusters:')
print(clusters)

Initializing Encoder from H: Full Matrix Size 128x384, Message Bits: 256
  > Inverting Parity Matrix (this may take a moment for large Z)...
  > Computing Generator Matrix...
Encoder Ready.
H shape: (128, 384), K=256, rate=0.667
Clusters:
[[  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15]
 [ 16  17  18  19  20  21  22  23  24  25  26  27  28  29  30  31]
 [ 32  33  34  35  36  37  38  39  40  41  42  43  44  45  46  47]
 [ 48  49  50  51  52  53  54  55  56  57  58  59  60  61  62  63]
 [ 64  65  66  67  68  69  70  71  72  73  74  75  76  77  78  79]
 [ 80  81  82  83  84  85  86  87  88  89  90  91  92  93  94  95]
 [ 96  97  98  99 100 101 102 103 104 105 106 107 108 109 110 111]
 [112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127]]


---
---

In [4]:
def generate_data(encoder, n_frames, snr_db, seed=42):
    rng = np.random.RandomState(seed)
    K, N = encoder.K, encoder.N
    messages = rng.randint(0, 2, size=(n_frames, K))
    codewords = encoder.encode(messages)
    bpsk = 1.0 - 2.0 * codewords.astype(np.float64)
    llr_matrix = AWGNChannel(bpsk, snr_db)
    return codewords, llr_matrix


TRAIN_SNR_DB = 4.0
N_TRAIN = 10000

train_codewords, train_llrs = generate_data(
    encoder, N_TRAIN, TRAIN_SNR_DB, seed=SEED
)

### Sanity Check

I initialized two agents. One agent i will train for 10,000 frames and another will not be trained at all. I will then compare the decoding performance of both the agents

In [5]:
bp_dec = BpDecoder(H, schedule="cluster")
env = PpoEnv(H, clusters, bp_dec, l_max=m)

agent = PpoAgent(
    obs_dim=n,
    num_clusters=len(clusters),
    lr=3e-4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_eps=0.2,
    ppo_epochs=4,
    minibatch_size=64,
    entropy_coeff=0.01,
    value_coeff=0.5,
)

In [6]:
N_TEST = 10000
TEST_SNRS = [3.0, 4.0, 5.0]
I_MAX = 10

In [7]:
ppo_dumb_bers = []
decoder_dumb = PpoDecoder(H, clusters, bp_dec, agent)

for snr_db in TEST_SNRS:
    test_cw, test_llrs = generate_data(
        encoder, N_TEST, snr_db, seed=SEED + 1
    )

    ppo_dumb_errors = 0
    for i in range(N_TEST):
        decoded = decoder_dumb.decode(
            test_llrs[i, :], I_MAX*len(clusters)
        )

        ppo_dumb_errors += np.sum(decoded != test_cw[i, :])
    
    ppo_dumb_ber = ppo_dumb_errors / (N_TEST * n)
    ppo_dumb_bers.append(ppo_dumb_ber)
    print(f"SNR={snr_db}dB: PPO Dumb BER = {ppo_dumb_ber:.4e}")

SNR=3.0dB: PPO Dumb BER = 3.2785e-02
SNR=4.0dB: PPO Dumb BER = 1.5279e-03
SNR=5.0dB: PPO Dumb BER = 7.5521e-06


### Training of agent

In [8]:
t0 = time.time()
rewards = agent.train(
    env,
    llr_list=[train_llrs[i] for i in range(N_TRAIN)],
    codeword_list=[train_codewords[i] for i in range(N_TRAIN)],
    update_every=10,
    verbose=True,
)
t_train = time.time() - t0
print(f"  Training time: {t_train:.1f}s")
print(f"  Final avg reward (last 50): {np.mean(rewards[-50:]):.4f}")

[PpoAgent] Training on device: cpu
  Episode 10/10000 | Avg reward: 45.4357 | π loss: -0.0153 | V loss: 73.0707 | entropy: 2.0764
  Episode 20/10000 | Avg reward: 56.3564 | π loss: -0.0104 | V loss: 68.8073 | entropy: 2.0749
  Episode 30/10000 | Avg reward: 30.0182 | π loss: -0.0118 | V loss: 46.1467 | entropy: 2.0743
  Episode 40/10000 | Avg reward: 36.1373 | π loss: -0.0158 | V loss: 44.9660 | entropy: 2.0742
  Episode 50/10000 | Avg reward: 55.4339 | π loss: -0.0143 | V loss: 51.2270 | entropy: 2.0720
  Episode 60/10000 | Avg reward: 43.5614 | π loss: -0.0179 | V loss: 36.4081 | entropy: 2.0691
  Episode 70/10000 | Avg reward: 36.2780 | π loss: -0.0129 | V loss: 56.0007 | entropy: 2.0680
  Episode 80/10000 | Avg reward: 57.6297 | π loss: -0.0050 | V loss: 49.8360 | entropy: 2.0706
  Episode 90/10000 | Avg reward: 41.3128 | π loss: -0.0194 | V loss: 47.2964 | entropy: 2.0645
  Episode 100/10000 | Avg reward: 33.0583 | π loss: -0.0403 | V loss: 24.9661 | entropy: 2.0667
  Episode 110/

In [10]:
ppo_bers = []
ppo_decoder = PpoDecoder(H, clusters, bp_dec, agent)

for snr_db in TEST_SNRS:
    test_cw, test_llrs = generate_data(
        encoder, N_TEST, snr_db, seed=SEED + 1
    )

    ppo_errors = 0
    for i in range(N_TEST):
        decoded = ppo_decoder.decode(
            test_llrs[i, :], I_MAX*len(clusters)
        )

        ppo_errors += np.sum(decoded != test_cw[i, :])
    
    ppo_ber = ppo_errors / (N_TEST * n)
    ppo_bers.append(ppo_ber)
    print(f"SNR={snr_db}dB: PPO BER = {ppo_ber:.4e}")

SNR=3.0dB: PPO BER = 3.2931e-02
SNR=4.0dB: PPO BER = 1.4581e-03
SNR=5.0dB: PPO BER = 1.1719e-05
